# Lab 2: Fine-Tuning with LoRA and Reinforcement Learning with Verifiable Rewards
**EECE.4860/5860 Introduction to Generative AI, Spring 2027**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acanets/genai-labs/blob/spring27/labs/lab2/lab2.ipynb)

In this lab you will:
* **Part A (all students):** fine-tune a small LLM to summarize dialogues with **LoRA**, count its trainable parameters, and measure ROUGE before and after fine-tuning.
* **Part B (all students):** compare the GPU memory of LoRA and **QLoRA** (4-bit base model).
* **Part C (EECE.5860 only):** write **verifiable reward functions** and train the model on arithmetic with **GRPO**, the RL algorithm behind DeepSeek-R1.

**Before you start:** *File > Save a copy in Drive*, then *Runtime > Change runtime type > T4 GPU*. Fill in the first code cell and run the cells in order.

**Time budget on a Colab T4:** about 15 min of GPU time for Parts A and B, plus about 10 min for Part C. Models: Qwen3-0.6B and Qwen3-1.7B (Apache-2.0). Dataset: DialogSum (Chen et al., 2021).

**Grading (15 points):** 2 for a notebook that runs top to bottom with outputs saved, 8 for the `[CHECK ...]` lines, and 5 for the report. EECE.5860: Part C adds 5 points and the total is scaled to 15.

**Submit on Canvas:** the notebook with outputs (*File > Download > Download .ipynb*) and a PDF of it (*File > Print > Save as PDF*). Do not edit the `[CHECK ...]` or `GENAI-RESULTS` lines.

## 0. Setup

In [ ]:
STUDENT_ID_LAST4 = ""   # last 4 digits of your student ID, e.g. "0427"
SECTION = "4860"        # "4860" or "5860"

In [ ]:
import importlib.util, os
if importlib.util.find_spec("genai_utils") is None:
    get_ipython().run_line_magic("pip", 'install -q "git+https://github.com/acanets/genai-labs@spring27"')
if importlib.util.find_spec("trl") is None or importlib.util.find_spec("rouge_score") is None:
    get_ipython().run_line_magic("pip", 'install -q "trl>=1.0" "peft>=0.17" rouge_score bitsandbytes')

import gc, json, math, random, re
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from genai_utils import setup_env, Checker, variant, measure

QUICK = os.environ.get("GENAI_QUICK") == "1"   # instructor smoke test only; leave as is
device, dtype = setup_env(seed=int(STUDENT_ID_LAST4 or 0))
ck = Checker("lab2", STUDENT_ID_LAST4, SECTION)
V = variant(STUDENT_ID_LAST4, 4)
LORA_RANK = [4, 8, 16, 32][V]                      # your personal LoRA rank
MODEL = "Qwen/Qwen3-0.6B"
FP16 = device == "cuda" and dtype == torch.float16
BF16 = device == "cuda" and dtype == torch.bfloat16
print(f"Your variant: {V}  ->  LoRA rank r = {LORA_RANK}")

In [ ]:
# Helpers (no changes needed)
def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

@torch.no_grad()
def summarize(model, tok, dialogues, max_new_tokens=64, batch_size=16):
    """Greedy summaries for a list of dialogues (thinking disabled)."""
    tok.padding_side = "left"
    model.eval()
    outs = []
    for i in range(0, len(dialogues), batch_size):
        prompts = [tok.apply_chat_template([{"role": "user", "content": INSTRUCTION + d}], tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)
                   for d in dialogues[i:i + batch_size]]
        enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += [s.strip() for s in tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    return outs

## Part A: LoRA fine-tuning for dialogue summarization (all students)

### A1. Data
DialogSum contains everyday conversations with human-written summaries. We fine-tune on 2,000 training dialogues and always evaluate on the **same 50 test dialogues**, so everyone's numbers are comparable.

**Task:** implement `to_prompt_completion(example)`, which turns one example into TRL's *conversational prompt-completion* format: `{"prompt": [user message], "completion": [assistant message]}`. TRL then computes the loss on the completion only.

In [ ]:
INSTRUCTION = "Summarize the following conversation in one or two sentences.\n\n"
raw = load_dataset("knkarthick/dialogsum")
train_raw = raw["train"].shuffle(seed=0).select(range(64 if QUICK else 2000))
test_raw = raw["test"].select(range(8 if QUICK else 50))
print(train_raw[0]["dialogue"][:300], "...\nSUMMARY:", train_raw[0]["summary"])

def to_prompt_completion(example):
    # TODO: your code here
    raise NotImplementedError

sample = to_prompt_completion({"dialogue": "#Person1#: Hi!\n#Person2#: Hello!", "summary": "They greet."})
ck.expect_true("A.format", isinstance(sample, dict) and set(sample) == {"prompt", "completion"}
               and sample["prompt"][0]["role"] == "user" and sample["completion"][0]["role"] == "assistant"
               and sample["prompt"][0]["content"].startswith(INSTRUCTION) and sample["completion"][0]["content"] == "They greet.")
train_ds = train_raw.map(to_prompt_completion, remove_columns=train_raw.column_names)

### A2. ROUGE and the baseline
ROUGE-N counts overlapping n-grams between a generated summary and a human reference (Lin, 2004); ROUGE-L uses the longest common subsequence. **Task:** implement `compute_rouge(preds, refs)`, returning the **mean F1** of `rouge1`, `rouge2`, and `rougeL` as a dict. Use `rouge_score.rouge_scorer.RougeScorer([...], use_stemmer=True)`.

In [ ]:
from rouge_score import rouge_scorer

def compute_rouge(preds, refs):
    # TODO: your code here
    raise NotImplementedError

ck.expect_fn("A.compute_rouge", lambda p, r: [round(v, 4) for v in compute_rouge(p, r).values()], [
    ((["the cat lay on the mat"], ["the cat sat on the mat"]), [0.8333, 0.6, 0.8333]),
    ((["a b", "x"], ["a b", "y"]), [0.5, 0.5, 0.5]),
], tol=1e-3)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype).to(device)
refs = test_raw["summary"]
with measure("baseline summaries"):
    base_preds = summarize(base, tok, test_raw["dialogue"])
rouge_before = compute_rouge(base_preds, refs)
print("Baseline example:", base_preds[0][:200])
print("ROUGE before fine-tuning:", {k: round(v, 4) for k, v in rouge_before.items()})
ck.record("A.rouge_before", {k: round(v, 4) for k, v in rouge_before.items()})
del base; free_memory()

### A3. How many parameters does LoRA train?
LoRA freezes a weight matrix $W \in \mathbb{R}^{d_\text{out} \times d_\text{in}}$ and learns $\Delta W = BA$ with $B \in \mathbb{R}^{d_\text{out} \times r}$, $A \in \mathbb{R}^{r \times d_\text{in}}$ (Hu et al., 2022). So each adapted matrix adds $r(d_\text{in} + d_\text{out})$ trainable parameters.

**Task:** implement `expected_lora_params(model, r, targets)`: loop over `model.named_modules()`, and for every `torch.nn.Linear` whose name ends with one of `targets`, add `r * (in_features + out_features)`. We compare your count with PEFT's.

In [ ]:
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]

def expected_lora_params(model, r, targets):
    # TODO: your code here
    raise NotImplementedError

lora_cfg = LoraConfig(r=LORA_RANK, lora_alpha=2 * LORA_RANK, lora_dropout=0.05, target_modules=TARGETS, task_type="CAUSAL_LM")
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype).to(device)
expected = expected_lora_params(model, LORA_RANK, TARGETS)

### A4. Fine-tune
The configuration below trains for 150 steps (about 2–4 minutes on a T4). Read each argument; you will discuss them in the report.

In [ ]:
OUT_DIR = "lab2_lora_adapter"   # tip: point this to a Google Drive folder to keep it after the session ends
args = SFTConfig(
    output_dir="lab2_sft", max_steps=4 if QUICK else 150, per_device_train_batch_size=8,
    gradient_accumulation_steps=1, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=5,
    logging_steps=10, max_length=512, fp16=FP16, bf16=BF16, report_to="none", save_strategy="no", seed=0,
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, processing_class=tok, peft_config=lora_cfg)
n_train = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in trainer.model.parameters())
print(f"Trainable: {n_train:,} of {n_all:,} parameters ({100 * n_train / n_all:.3f}%). Your formula: {expected:,}")
ck.expect_true("A.lora_param_count", expected == n_train, value=(expected, n_train))
ck.record("A.trainable_percent", round(100 * n_train / n_all, 4))
lora_stats = {}
with measure("LoRA fine-tuning", lora_stats):
    trainer.train()
ck.record("A.train_seconds", lora_stats["seconds"])
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
print("training loss:", [round(l, 3) for l in losses])
trainer.model.save_pretrained(OUT_DIR)
ck.expect_true("A.adapter_saved", os.path.exists(os.path.join(OUT_DIR, "adapter_config.json")))

In [ ]:
with measure("fine-tuned summaries"):
    ft_preds = summarize(trainer.model, tok, test_raw["dialogue"])
rouge_after = compute_rouge(ft_preds, refs)
print(f"{'':10s}{'rouge1':>9s}{'rouge2':>9s}{'rougeL':>9s}")
for name, r in (("before", rouge_before), ("after", rouge_after)):
    print(f"{name:10s}" + "".join(f"{r[k]:9.4f}" for k in ("rouge1", "rouge2", "rougeL")))
for i in range(2):
    print(f"\nREFERENCE: {refs[i]}\nBEFORE:    {base_preds[i][:200]}\nAFTER:     {ft_preds[i][:200]}")
ck.record("A.rouge_after", {k: round(v, 4) for k, v in rouge_after.items()})
ck.expect_true("A.rouge_improved", rouge_after["rougeL"] > rouge_before["rougeL"],
               value=(round(rouge_before["rougeL"], 4), round(rouge_after["rougeL"], 4)))
del trainer, model; free_memory()

## Part B: LoRA vs. QLoRA memory (all students)
QLoRA trains LoRA adapters on a base model stored in 4-bit NF4 (Dettmers et al., 2023). We train Qwen3-1.7B for 20 steps both ways and record peak GPU memory and time per step. This part needs a CUDA GPU (bitsandbytes).

**Task:** create `bnb_config`, a `BitsAndBytesConfig` that loads the model in 4-bit NF4 with double quantization and float16 compute.

In [ ]:
from transformers import BitsAndBytesConfig
BIG = "Qwen/Qwen3-1.7B"

def short_run(quantization_config=None):
    tok_b = AutoTokenizer.from_pretrained(BIG)
    m = AutoModelForCausalLM.from_pretrained(BIG, dtype=torch.float16, quantization_config=quantization_config,
                                             device_map={"": 0})
    a = SFTConfig(output_dir="lab2_mem", max_steps=20, per_device_train_batch_size=4, learning_rate=2e-4,
                  max_length=512, fp16=True, report_to="none", save_strategy="no", logging_steps=10,
                  gradient_checkpointing=True)
    t = SFTTrainer(model=m, args=a, train_dataset=train_ds, processing_class=tok_b,
                   peft_config=LoraConfig(r=16, lora_alpha=32, target_modules=TARGETS, task_type="CAUSAL_LM"))
    stats = {}
    with measure("20 training steps", stats):
        t.train()
    del t, m; free_memory()
    return stats

if torch.cuda.is_available() and not QUICK:
    # TODO: your code here
    raise NotImplementedError
    lora_mem = short_run(None)
    qlora_mem = short_run(bnb_config)
    print(f"{'':8s}{'peak GiB':>10s}{'s/step':>9s}")
    for name, s in (("LoRA", lora_mem), ("QLoRA", qlora_mem)):
        print(f"{name:8s}{s['peak_mem_gb']:10.2f}{s['seconds'] / 20:9.2f}")
    ck.record("B.lora", lora_mem); ck.record("B.qlora", qlora_mem)
    ck.expect_true("B.qlora_uses_less_memory", qlora_mem["peak_mem_gb"] < lora_mem["peak_mem_gb"],
                   value=(lora_mem["peak_mem_gb"], qlora_mem["peak_mem_gb"]))
else:
    print("Part B needs a CUDA GPU; skipped.")

## Part C: RL with verifiable rewards (EECE.5860 only; EECE.4860 students may skip to the report)
Reasoning models such as DeepSeek-R1 were trained with **rule-based rewards**: a program checks whether the answer is correct and correctly formatted, so no learned reward model is needed (DeepSeek-AI, 2025). **GRPO** samples a group of completions per prompt and pushes the policy toward the completions whose reward is above the group average (Shao et al., 2024).

The task is arithmetic. The model must end with the final number inside `<answer>...</answer>`.

**Task 1:** write two reward functions. Each receives a list of completion strings (plus dataset columns as keyword arguments) and returns one float per completion:
* `format_reward`: 1.0 if the completion contains **exactly one** `<answer>...</answer>` block whose content is an integer (optionally negative, surrounding spaces allowed), else 0.0;
* `correctness_reward`: 1.0 if that integer equals the reference `answer` (a string column), else 0.0.

During training, TRL passes chat-formatted completions (lists of messages), so start each function with `texts = [as_text(c) for c in completions]`.

In [ ]:
ANSWER_RE = re.compile(r"<answer>\s*(-?\d+)\s*</answer>")

def as_text(c):
    """Completion as a plain string (TRL passes [{"role": "assistant", "content": ...}] for chat prompts)."""
    return c[0]["content"] if isinstance(c, list) else c

def format_reward(completions, **kwargs):
    # TODO: your code here
    raise NotImplementedError

def correctness_reward(completions, answer, **kwargs):
    # TODO: your code here
    raise NotImplementedError

if SECTION == "5860":
    ck.expect_fn("C.format_reward", lambda c: format_reward(c), [
        ((["Thinking... <answer>42</answer>"],), [1.0]),
        ((["<answer> -7 </answer>"],), [1.0]),
        ((["The answer is 42."],), [0.0]),
        ((["<answer>4.5</answer>"],), [0.0]),
        ((["<answer>1</answer> or <answer>2</answer>"],), [0.0]),
        ((["<answer>12</answer>", "nope"],), [1.0, 0.0]),
    ])
    ck.expect_fn("C.correctness_reward", lambda c, a: correctness_reward(c, answer=a), [
        ((["<answer>42</answer>"], ["42"]), [1.0]),
        ((["<answer>41</answer>"], ["42"]), [0.0]),
        ((["42"], ["42"]), [0.0]),
        ((["<answer>-3</answer>", "<answer>5</answer>"], ["-3", "6"]), [1.0, 0.0]),
        ((["<answer>1</answer><answer>42</answer>"], ["42"]), [0.0]),
    ])

**Task 2:** run GRPO (about 6–8 minutes on a T4) and plot the mean reward. Then compare accuracy on 40 held-out problems before and after training.

In [ ]:
def make_problems(n, seed):
    rng = random.Random(seed)
    rows = []
    for _ in range(n):
        a, b, c = rng.randint(2, 40), rng.randint(2, 12), rng.randint(2, 12)
        op = rng.choice(["+", "-"])
        q = f"What is {a} {op} {b} * {c}? Reason briefly, then give only the final integer inside <answer></answer> tags."
        rows.append({"prompt": [{"role": "user", "content": q}], "answer": str(a + b * c if op == "+" else a - b * c)})
    return Dataset.from_list(rows)

@torch.no_grad()
def accuracy(model, tok, ds, max_new_tokens=160):
    tok.padding_side = "left"
    prompts = [tok.apply_chat_template(r["prompt"], tokenize=False, add_generation_prompt=True, enable_thinking=False) for r in ds]
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    texts = tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return sum(correctness_reward(texts, answer=ds["answer"])) / len(ds), sum(format_reward(texts)) / len(ds)

if SECTION == "5860":
    from trl import GRPOConfig, GRPOTrainer
    train_rl, eval_rl = make_problems(400, seed=V), make_problems(8 if QUICK else 40, seed=1000)
    policy = AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype).to(device)
    acc_before = accuracy(policy, tok, eval_rl)
    grpo_args = GRPOConfig(
        output_dir="lab2_grpo", max_steps=3 if QUICK else 60, per_device_train_batch_size=8, num_generations=4,
        max_completion_length=96 if QUICK else 160, learning_rate=5e-5, temperature=1.0, logging_steps=1 if QUICK else 5,
        fp16=FP16, bf16=BF16, report_to="none", save_strategy="no", seed=0,
        chat_template_kwargs={"enable_thinking": False},
    )
    grpo = GRPOTrainer(model=policy, reward_funcs=[format_reward, correctness_reward], args=grpo_args,
                       train_dataset=train_rl, processing_class=tok,
                       peft_config=LoraConfig(r=16, lora_alpha=32, target_modules=TARGETS, task_type="CAUSAL_LM"))
    with measure("GRPO training"):
        grpo.train()
    acc_after = accuracy(grpo.model, tok, eval_rl)
    log = [h for h in grpo.state.log_history if "reward" in h]
    rewards = [h["reward"] for h in log]
    print("mean reward per logging step:", [round(r, 2) for r in rewards])
    print(f"held-out accuracy / format rate: before {acc_before}, after {acc_after}")
    try:
        import matplotlib.pyplot as plt
        plt.plot([h["step"] for h in log], rewards, marker="o"); plt.xlabel("step"); plt.ylabel("mean reward")
        plt.title("GRPO training reward"); plt.show()
    except ImportError:
        pass
    ck.record("C.rewards", [round(r, 3) for r in rewards])
    ck.record("C.accuracy_before_after", {"before": acc_before, "after": acc_after})
    k = max(1, len(rewards) // 3)
    ck.expect_true("C.reward_increased", sum(rewards[-k:]) / k > sum(rewards[:k]) / k,
                   value=(round(sum(rewards[:k]) / k, 3), round(sum(rewards[-k:]) / k, 3)))

## Results summary
Run this cell last. **Do not edit its output.**

In [ ]:
_ = ck.summary()

## Report (5 points)
Answer in the Markdown cells below (2–5 sentences each), citing your own numbers.

### 1. Summary
**Your answer:**

### 2. Problems encountered and how you solved them
**Your answer:**

### 3. Questions
**Q1 (A3).** Your LoRA rank was r = ? What percentage of the model's parameters did you train? Show how one `q_proj` matrix contributes to the count.

**Your answer:**

**Q2 (A4).** Compare ROUGE before and after fine-tuning, and quote one before/after pair. Did fine-tuning change *what* the model says, *how* it says it (length and style), or both?

**Your answer:**

**Q3 (A4).** Name two limitations of ROUGE as a measure of summary quality (Lin, 2004; see Lecture 5). Give a case from your outputs where ROUGE and your own judgment disagree, if you find one.

**Your answer:**

**Q4 (B).** Report peak memory and time per step for LoRA and QLoRA. Explain where the memory saving comes from and why QLoRA may be slower.

**Your answer:**

**Q5 (C, EECE.5860).** How did the mean reward and the held-out accuracy change during GRPO? Which reward (format or correctness) improved first, and why might that be?

**Your answer:**

**Q6 (C, EECE.5860).** Describe one way your reward functions could be *hacked* (rewarding an output that is not what you want) and how you would close that loophole.

**Your answer:**

### 4. Lessons learned
**Your answer:**

### 5. AI-use disclosure
*Follow the course AI-use policy in the syllabus; write "None" if you used no AI tools.*

**Your answer:**